'''
Source: Impact4cast (Max Planck Institute)
Original code from the Max Planck Institute for Informatics / Max Planck Institute for Security and Privacy research group.

Modifications: Bug fixes for checkpoint resume mechanism and dataset adaptation for scientific entity impact prediction.
'''


In [ ]:
import os
from datetime import datetime, date
import random, time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from scipy import sparse
from collections import defaultdict
import pandas as pd
import networkx as nx
import copy
import gzip
import pickle
from scipy.stats import rankdata
import time

### single concept's citation features

In [ ]:
time_start = time.time()
data_folder="data_concept_graph"
concept_folder="concept_folder"
# # Read all concepts together with time, citation information
dynamic_concept_file=os.path.join(data_folder,"full_dynamic_concept.parquet")
full_concepts_dynamic_data = pd.read_parquet(dynamic_concept_file)

# Read all concepts from full_concepts_for_openalex.txt
concepts_files = os.path.join(concept_folder, 'final_concepts.txt')
with open(concepts_files, 'r') as file:
    full_concepts = [concept.strip() for concept in file.readlines()]

print(f"Done, elapsed_time: {time.time() - time_start}\n full_concepts_dynamic_data: {len(full_concepts_dynamic_data)};\n full_concept: {len(full_concepts)}")


In [ ]:
NUM_OF_VERTICES=len(full_concepts)
vertex_degree_cutoff=1
years_delta=3
min_edges=1

In [ ]:
# 查看数据中的列
print("数据列:", full_concepts_dynamic_data.columns.tolist())

# 查看年份列
year_cols = [col for col in full_concepts_dynamic_data.columns if col.startswith('c') and col[1:].isdigit()]
print("年份列:", year_cols)
print("年份范围:", [int(col[1:]) for col in year_cols])

# 查看数据形状
print("数据形状:", full_concepts_dynamic_data.shape)

In [ ]:

years=[2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

day_origin = date(1990,1,1)
all_concepts_df = pd.DataFrame({'v1': range(0, NUM_OF_VERTICES)})

store_folder="data_for_features"
if not os.path.exists(store_folder):
    os.makedirs(store_folder)

start_time=time.time()
for yy in years:  
    print(f'Year: {yy}')
    day_curr=(date(yy,12,31)- day_origin).days
    columns_to_subtract = [f'c{i}' for i in range(2023, yy, -1)]
    print(columns_to_subtract)
    cols_to_sum = [f'c{i}' for i in range(yy, yy-years_delta, -1)]
    print(cols_to_sum)
    
    dynamic_concepts=full_concepts_dynamic_data[full_concepts_dynamic_data['time']<=day_curr]
    dynamic_concepts_df = dynamic_concepts.copy()
    
    dynamic_concepts_df[f'ct_{yy}'] = dynamic_concepts_df['ct'] - dynamic_concepts_df[columns_to_subtract].sum(axis=1)
    
    dynamic_concepts_df['ct_delta'] = dynamic_concepts_df[cols_to_sum].sum(axis=1)
    
    dynamic_concepts_df=dynamic_concepts_df[['v1', f'c{yy}', f'ct_{yy}', 'ct_delta']]
    
    dynamic_concepts_grouped = dynamic_concepts_df.groupby('v1').agg({f'c{yy}':'sum', f'ct_{yy}':'sum', 'ct_delta':'sum', 'v1':'size'}).rename(columns={'v1':f'num'}).reset_index()
    
    dynamic_concepts_grouped[f'c{yy}_m'] = dynamic_concepts_grouped[f'c{yy}'] / dynamic_concepts_grouped[f'num']
    dynamic_concepts_grouped[f'ct_{yy}_m'] = dynamic_concepts_grouped[f'ct_{yy}'] / dynamic_concepts_grouped[f'num']
    dynamic_concepts_grouped[f'ct_delta_m'] = dynamic_concepts_grouped['ct_delta'] / dynamic_concepts_grouped[f'num']
     
    
    # Merge with all_concepts_df
    dynamic_concepts_data = pd.merge(all_concepts_df, dynamic_concepts_grouped, on='v1', how='left')
    dynamic_concepts_data.fillna(0, inplace=True) # Fill NaN values with 0
    dynamic_concepts_data.sort_values(by='v1')
    
    data_file = os.path.join(store_folder, f"concept_node_citation_data_{yy}.parquet")
    dynamic_concepts_data.to_parquet(data_file, compression='gzip')
    print(f"in {yy}; time: {time.time()-start_time}\n")
    start_time=time.time()


In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import date

# 配置参数
years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
years_delta = 3  # 时间窗口大小
day_origin = date(1990, 1, 1)

# 假设这些变量已在前面定义
# NUM_OF_VERTICES - 概念总数
# full_concepts_dynamic_data - 动态概念数据DataFrame

# 创建存储文件夹
store_folder = "data_for_features"
if not os.path.exists(store_folder):
    os.makedirs(store_folder)

# 所有概念的DataFrame
all_concepts_df = pd.DataFrame({'v1': range(0, NUM_OF_VERTICES)})

# 查看数据中的可用列（调试用）
print("="*60)
print("数据信息")
print("="*60)
print(f"数据形状: {full_concepts_dynamic_data.shape}")
print(f"数据列: {full_concepts_dynamic_data.columns.tolist()}")

# 找出所有年份列
year_cols = [col for col in full_concepts_dynamic_data.columns 
             if col.startswith('c') and col[1:].isdigit()]
available_years = sorted([int(col[1:]) for col in year_cols])
print(f"数据中可用的年份列: {available_years}")
print(f"最大可用年份: {max(available_years) if available_years else '无'}")
print("="*60)

start_time = time.time()
processed_years = []

for yy in years:
    print(f'\n处理年份: {yy}')
    chunk_start = time.time()
    
    # 计算时间阈值（该年最后一天）
    day_curr = (date(yy, 12, 31) - day_origin).days
    
    # 筛选该年份之前的数据
    dynamic_concepts = full_concepts_dynamic_data[full_concepts_dynamic_data['time'] <= day_curr].copy()
    print(f"  筛选后数据量: {len(dynamic_concepts)} 行")
    
    if len(dynamic_concepts) == 0:
        print(f"  警告: {yy}年没有数据，跳过")
        continue
    
    # ========== 1. 处理要减去的列（未来年份的引用） ==========
    # 从最大可用年份到yy+1（只取存在的列）
    max_year = max(available_years) if available_years else yy
    future_years = [y for y in range(max_year, yy, -1) if y in available_years]
    columns_to_subtract = [f'c{y}' for y in future_years]
    
    print(f"  要减去的未来年份: {future_years}")
    
    # 计算到该年份为止的累积引用（排除未来年份）
    if columns_to_subtract:
        # 检查这些列是否都存在
        missing_cols = [col for col in columns_to_subtract if col not in dynamic_concepts.columns]
        if missing_cols:
            print(f"  警告: 以下列不存在: {missing_cols}")
            columns_to_subtract = [col for col in columns_to_subtract if col in dynamic_concepts.columns]
        
        if columns_to_subtract:
            dynamic_concepts[f'ct_{yy}'] = dynamic_concepts['ct'] - dynamic_concepts[columns_to_subtract].sum(axis=1)
        else:
            dynamic_concepts[f'ct_{yy}'] = dynamic_concepts['ct']
    else:
        dynamic_concepts[f'ct_{yy}'] = dynamic_concepts['ct']
    
    # ========== 2. 处理要加总的列（历史窗口内的引用） ==========
    # 从yy到yy-years_delta（只取存在的列）
    past_years = [y for y in range(yy, yy - years_delta, -1) if y in available_years]
    cols_to_sum = [f'c{y}' for y in past_years]
    
    print(f"  要加总的历史年份: {past_years}")
    
    # 计算时间窗口内的引用总和
    if cols_to_sum:
        # 检查这些列是否都存在
        missing_cols = [col for col in cols_to_sum if col not in dynamic_concepts.columns]
        if missing_cols:
            print(f"  警告: 以下列不存在: {missing_cols}")
            cols_to_sum = [col for col in cols_to_sum if col in dynamic_concepts.columns]
        
        if cols_to_sum:
            dynamic_concepts['ct_delta'] = dynamic_concepts[cols_to_sum].sum(axis=1)
        else:
            dynamic_concepts['ct_delta'] = 0
    else:
        dynamic_concepts['ct_delta'] = 0
    
    # ========== 3. 确保当前年份列存在 ==========
    c_col = f'c{yy}'
    if c_col not in dynamic_concepts.columns:
        print(f"  警告: {c_col} 列不存在，创建空列")
        dynamic_concepts[c_col] = 0
    
    # ========== 4. 选择需要的列 ==========
    required_cols = ['v1', c_col, f'ct_{yy}', 'ct_delta']
    existing_required = [col for col in required_cols if col in dynamic_concepts.columns]
    
    if len(existing_required) < len(required_cols):
        missing = set(required_cols) - set(existing_required)
        print(f"  错误: 缺少必要列: {missing}")
        continue
    
    dynamic_concepts_subset = dynamic_concepts[required_cols].copy()
    
    # ========== 5. 按概念ID聚合 ==========
    print(f"  开始聚合...")
    agg_dict = {
        c_col: 'sum',
        f'ct_{yy}': 'sum',
        'ct_delta': 'sum',
        'v1': 'size'
    }
    
    dynamic_concepts_grouped = dynamic_concepts_subset.groupby('v1').agg(agg_dict).rename(columns={'v1': 'num'}).reset_index()
    
    # ========== 6. 计算平均值 ==========
    # 避免除零错误
    dynamic_concepts_grouped['num'] = dynamic_concepts_grouped['num'].replace(0, 1)
    
    dynamic_concepts_grouped[f'{c_col}_m'] = dynamic_concepts_grouped[c_col] / dynamic_concepts_grouped['num']
    dynamic_concepts_grouped[f'ct_{yy}_m'] = dynamic_concepts_grouped[f'ct_{yy}'] / dynamic_concepts_grouped['num']
    dynamic_concepts_grouped['ct_delta_m'] = dynamic_concepts_grouped['ct_delta'] / dynamic_concepts_grouped['num']
    
    # ========== 7. 与所有概念合并 ==========
    print(f"  合并数据...")
    dynamic_concepts_data = pd.merge(all_concepts_df, dynamic_concepts_grouped, on='v1', how='left')
    dynamic_concepts_data.fillna(0, inplace=True)
    dynamic_concepts_data.sort_values(by='v1', inplace=True)
    
    # ========== 8. 保存结果 ==========
    data_file = os.path.join(store_folder, f"concept_node_citation_data_{yy}.parquet")
    dynamic_concepts_data.to_parquet(data_file, compression='gzip')
    
    processed_years.append(yy)
    
    # 打印统计信息
    elapsed = time.time() - chunk_start
    total_elapsed = time.time() - start_time
    print(f"  完成 {yy}")
    print(f"    聚合后概念数: {len(dynamic_concepts_grouped)}")
    print(f"    最终特征维度: {dynamic_concepts_data.shape}")
    print(f"    耗时: {elapsed:.2f}秒")
    print(f"    累计耗时: {total_elapsed:.2f}秒")

# ========== 最终统计 ==========
print("\n" + "="*60)
print("处理完成！")
print("="*60)
print(f"成功处理的年份: {processed_years}")
print(f"总耗时: {time.time() - start_time:.2f}秒")
print(f"输出文件夹: {store_folder}")

# 列出生成的文件
print("\n生成的文件:")
for file in sorted(os.listdir(store_folder)):
    if file.endswith('.parquet'):
        file_path = os.path.join(store_folder, file)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
        print(f"  {file}: {file_size:.2f} MB")

print("="*60)

### concept pair's citation features

In [ ]:
time_start = time.time()
data_folder="data_concept_graph"

# Read all concepts together with time, citation information
graph_file=os.path.join(data_folder,"full_dynamic_graph.parquet")
full_edge_dynamic_data = pd.read_parquet(graph_file)

print(f"Done, elapsed_time: {time.time() - time_start}\n full_edge_dynamic_data: {len(full_edge_dynamic_data)};\n")


In [ ]:

years=[2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022,2023,2024, 2025]

day_origin = date(1990,1,1)
 
store_folder="data_for_features"
start_time=time.time()
for yy in years:  
    print(f'Year: {yy}')
    day_curr=(date(yy,12,31)- day_origin).days
    columns_to_subtract = [f'c{i}' for i in range(2023, yy, -1)]
    print(columns_to_subtract)
    cols_to_sum = [f'c{i}' for i in range(yy, yy-years_delta, -1)]
    print(cols_to_sum)
    
    dynamic_pairs=full_edge_dynamic_data[full_edge_dynamic_data['time']<=day_curr]
    dynamic_pairs_df = dynamic_pairs.copy()
    
    dynamic_pairs_df[f'ct_{yy}'] = dynamic_pairs_df['ct'] - dynamic_pairs_df[columns_to_subtract].sum(axis=1)
    
    dynamic_pairs_df['ct_delta'] = dynamic_pairs_df[cols_to_sum].sum(axis=1)
    
    dynamic_pairs_df=dynamic_pairs_df[['v1', 'v2', f'c{yy}', f'ct_{yy}', 'ct_delta']]
    
    dynamic_pairs_grouped = dynamic_pairs_df.groupby(['v1','v2']).agg({f'c{yy}':'sum', f'ct_{yy}':'sum', 'ct_delta':'sum', 'v1':'size'}).rename(columns={'v1':f'num'}).reset_index()
    
    dynamic_pairs_grouped[f'c{yy}_m'] = dynamic_pairs_grouped[f'c{yy}'] / dynamic_pairs_grouped[f'num']
    dynamic_pairs_grouped[f'ct_{yy}_m'] = dynamic_pairs_grouped[f'ct_{yy}'] / dynamic_pairs_grouped[f'num']
    dynamic_pairs_grouped[f'ct_delta_m'] = dynamic_pairs_grouped['ct_delta'] / dynamic_pairs_grouped[f'num']
    
    data_file = os.path.join(store_folder, f"concept_pair_citation_data_{yy}.parquet")
    dynamic_pairs_grouped.to_parquet(data_file, compression='gzip')
    print(f"in {yy}; time: {time.time()-start_time}\n")
    start_time=time.time()
    